# Creating Real DAGs in Airflow


## Prerequisites

- Finish **`01_airflow_basics.ipynb`**.
- Start Airflow from **[`airflow/README.md`](../../airflow/README.md)**.


## Why DAG design matters

Real pipelines are **not** single tasks.

They involve:
- Multiple steps
- Dependencies
- Data flow

👉 Poor DAG design → fragile pipelines
👉 Good DAG design → reliable systems


## Typical data pipeline DAG

**Extract → Transform → Load**

Each step is:
👉 **A task**

Tasks are connected using **dependencies**.


## Multi-task DAG

The repo includes **`../../airflow/dags/etl_dag.py`** — mounted into the container as `/opt/airflow/dags/`.

```python
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

def extract():
    print("Extracting data...")

def transform():
    print("Transforming data...")

def load():
    print("Loading data...")

with DAG(
    dag_id="etl_simple_chain_dag",
    start_date=datetime(2024, 1, 1),
    schedule_interval="@daily",
    catchup=False,
) as dag:

    task_extract = PythonOperator(
        task_id="extract",
        python_callable=extract,
    )

    task_transform = PythonOperator(
        task_id="transform",
        python_callable=transform,
    )

    task_load = PythonOperator(
        task_id="load",
        python_callable=load,
    )

    task_extract >> task_transform >> task_load
```

Enable **`etl_simple_chain_dag`** in the UI, trigger it, and open **Graph** view.


## Parallel tasks

Fan out from one task to several, then join:

```python
def validate():
    print("Validating data...")

task_validate = PythonOperator(
    task_id="validate",
    python_callable=validate,
)

# Parallel flow
task_extract >> [task_transform, task_validate] >> task_load
```


## Task dependencies

**Linear:**
`A → B → C`

**Parallel:**
`A → [B, C] → D`

👉 You control execution flow; keep the graph **acyclic**.


## Passing data between tasks

Return values from a `PythonOperator` are stored as **XCom**. Downstream tasks use **`ti.xcom_pull`**.

```python
def extract():
    return {"data": [1, 2, 3]}

def transform(ti):
    data = ti.xcom_pull(task_ids="extract")
    print("Received:", data)

task_extract = PythonOperator(
    task_id="extract",
    python_callable=extract,
)

task_transform = PythonOperator(
    task_id="transform",
    python_callable=transform,
)
```

👉 **XCom** = cross-communication for small payloads (not bulk data).


## Real pipeline thinking

**Extract API → Clean data → Store in DB → Run analytics**

👉 Each stage is a task (or task group).
👉 The DAG defines **order** and **parallelism**.


## Practice

1. DAG with **`extract`**, **`clean`**, **`load`**.
2. Add a **`validation`** task.
3. Run validation **in parallel** with clean before load.


## Assignment

Tasks:
- **`extract_users`**
- **`transform_users`**
- **`load_users`**
- **`validate_data`**

**Flow:**
`extract_users → [transform_users, validate_data] → load_users`

**Bonus:** logging in each task — reference: **`../../airflow/dags/users_pipeline_dag.py`**.


## Interview questions

1. What is a DAG?
2. How do you manage dependencies?
3. What is XCom?
4. How do tasks communicate?


## What you just learned

- Multi-step workflows
- Task dependencies
- Parallel execution
- Passing data with **XCom**

👉 This is how production pipelines are structured.


---

**Next:** **Scheduling + retries + failure handling** — automatic runs, retries, and production readiness.


**Reality check:** you can write transforms in Python/SQL/Spark — orchestration is how they **run on a schedule** with **clear dependencies** and **supervision**.


## Your tasks

- [ ] **`etl_dag.py`:** enable **`etl_simple_chain_dag`**, trigger, verify **linear** logs.
- [ ] **`users_pipeline_dag.py`:** enable **`users_pipeline_dag`**, confirm **parallel** branches + **XCom** in logs.
- [ ] **Practice:** build your own DAG with **`clean`** parallel to **`validation`**.
- [ ] Skim interview questions (short written answers optional).
